In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

super_ai_engineer_season_6_individual_hackathon_house_recognition_path = kagglehub.competition_download('super-ai-engineer-season-6-individual-hackathon-house-recognition')

print('Data source import complete.')


# SECTION 1: Imports & Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset, random_split
from transformers import ViTForImageClassification, ViTImageProcessor, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score

# ตั้ง seed เพื่อให้ผลลัพธ์ reproducible
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# เลือก device อัตโนมัติ (ต้องกำหนดก่อน model)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


# SECTION 2:Config

In [ ]:
MODEL_NAME    = "google/vit-base-patch16-224"
TRAIN_CSV     = "/kaggle/input/competitions/super-ai-engineer-season-6-individual-hackathon-house-recognition/train.csv"
TRAIN_FOLDER  = "/kaggle/input/competitions/super-ai-engineer-season-6-individual-hackathon-house-recognition/train/train"
TEST_FOLDER   = "/kaggle/input/competitions/super-ai-engineer-season-6-individual-hackathon-house-recognition/test/test"
OUTPUT_DIR    = "./results"
SUBMISSION    = "./submission.csv"

NUM_LABELS    = 2
BATCH_SIZE    = 16
NUM_EPOCHS    = 5
LEARNING_RATE = 2e-5
TRAIN_RATIO   = 0.8

# SECTION 3: Load Model & Processor

In [ ]:
feature_extractor = ViTImageProcessor.from_pretrained(MODEL_NAME)

model = ViTForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    ignore_mismatched_sizes=True  # classifier head จะถูก reinit เป็น 2 classes
)
model = model.to(device)  # แก้ไข: move model ไป GPU ก่อนเริ่ม

print(f"Classifier head: {model.classifier}")
# ควรแสดง: Linear(in_features=768, out_features=2, bias=True)

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                        
------------------+----------+----------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 768]) vs model:torch.Size([2, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([2])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Classifier head: Linear(in_features=768, out_features=2, bias=True)


# SECTION 4 : Dataset Class

In [ ]:
class HouseDataset(Dataset):
    """Dataset สำหรับโหลดรูปบ้านและ label จาก DataFrame"""

    def __init__(self, df: pd.DataFrame, image_folder: str, feature_extractor):
        self.df = df.reset_index(drop=True)
        self.image_folder = image_folder
        self.feature_extractor = feature_extractor

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> dict:
        row = self.df.iloc[idx]
        image_path = os.path.join(self.image_folder, row["image_name"])

        # แก้ไข: เพิ่ม error handling กรณีเปิดรูปไม่ได้
        try:
            image = Image.open(image_path).convert("RGB")
        except Exception as e:
            print(f"[WARNING] Cannot open image: {image_path} — {e}")
            image = Image.new("RGB", (224, 224), color=(128, 128, 128))

        inputs = self.feature_extractor(images=image, return_tensors="pt")
        label  = torch.tensor(row["class"], dtype=torch.long)

        return {
            "pixel_values": inputs["pixel_values"].squeeze(0),
            "labels": label
        }

# SECTION 5: Load Data & Split

In [ ]:
df = pd.read_csv(TRAIN_CSV)
print(f"Total samples: {len(df)}")
print(f"Class distribution:\n{df['class'].value_counts()}")

# ตรวจสอบ missing values
print(f"\nMissing values:\n{df.isnull().sum()}")

full_dataset = HouseDataset(df, TRAIN_FOLDER, feature_extractor)

train_size = int(TRAIN_RATIO * len(full_dataset))
eval_size  = len(full_dataset) - train_size
train_dataset, eval_dataset = random_split(
    full_dataset, [train_size, eval_size],
    generator=torch.Generator().manual_seed(SEED)
)
print(f"\nTrain: {train_size} | Eval: {eval_size}")

Total samples: 2953
Class distribution:
class
0    1520
1    1433
Name: count, dtype: int64

Missing values:
image_name    0
class         0
dtype: int64

Train: 2362 | Eval: 591


# SECTION 6: Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",         # แก้ไข: evaluation_strategy → eval_strategy (deprecated ใน v5)
    save_strategy="epoch",
    load_best_model_at_end=True,   # เพิ่ม: โหลด best checkpoint หลัง train เสร็จ
    metric_for_best_model="f1",    # เพิ่ม: ใช้ F1 เป็น criteria แทน loss
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_dir="./logs",
    logging_steps=20,
    fp16=torch.cuda.is_available(), # เพิ่ม: Mixed precision ถ้ามี GPU → เร็วขึ้น ~2x
    report_to="none",
    seed=SEED,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


# SECTION 7: Metrics

In [ ]:
def compute_metrics(eval_pred) -> dict:
    """คำนวณ Accuracy และ F1 Score"""
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1":       f1_score(labels, preds, average="binary"),  # เพิ่ม F1
    }

# SECTION 8: Trainer & Train

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


# SECTION 9: Inference & Submission

In [ ]:
def predict_and_submit(
    test_folder: str,
    model,
    feature_extractor,
    device,
    output_path: str = "./submission.csv"
) -> pd.DataFrame:
    """
    รันการทำนายทุกรูปใน test_folder
    แล้ว export submission.csv
    """
    model.eval()
    model.to(device)

    image_files = sorted([
        f for f in os.listdir(test_folder)
        if f.lower().endswith((".jpg", ".png"))
    ])

    results = []
    for image_file in image_files:
        image_path = os.path.join(test_folder, image_file)

        try:
            image = Image.open(image_path).convert("RGB")
        except Exception as e:
            print(f"[WARNING] Cannot open: {image_path} — {e}")
            results.append({"image_name": image_file, "class": 0})
            continue

        inputs = feature_extractor(images=image, return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = model(**inputs)
            predicted_label = torch.argmax(outputs.logits, dim=-1).item()

        results.append({"image_name": image_file, "class": predicted_label})

    submission_df = pd.DataFrame(results)
    submission_df.to_csv(output_path, index=False)
    print(f"\nSaved {len(submission_df)} predictions → {output_path}")
    return submission_df

In [ ]:
# รัน predict
submission = predict_and_submit(TEST_FOLDER, model, feature_extractor, device, SUBMISSION)
print(submission.head(10))
print(f"\nClass distribution in predictions:\n{submission['class'].value_counts()}")